In [ ]:

import os
import pandas as pd
from openai import OpenAI
import time
from collections import Counter

# Set your API key as an environment variable
os.environ["OPENAI_API_KEY"] = "xx"
client = OpenAI()

# Define the system prompt for consistent SDG analysis
SYSTEM_PROMPT = """
You are a meticulous expert in classifying university courses according to the UN Sustainable Development Goals (SDGs). Your task is to analyze each course description and determine which of the 17 SDGs are explicitly and directly addressed.

Instructions:
1. **Evidence-Based Classification:** Only assign SDGs that are clearly and directly supported by specific content in the course description. Avoid assumptions or interpretations based on vague, indirect, or general statements.
2. **Explicit Alignment Required:** The course must substantively focus on issues that align with specific SDG targets, themes, or language.
3. **Equal Consideration:** Evaluate alignment across all 17 SDGs without bias toward specific goals.
4. **No Clear Match:** If the course does not provide sufficient evidence for alignment with any SDG, return “None”.

Output Format (strictly adhere to this structure):
SDGs: [List of applicable SDG numbers separated by commas (e.g., 4, 5, 13)], or “None” if no SDG applies.  
Justification: [1–2 concise sentences citing direct phrases, topics, or keywords from the course description that support the chosen SDGs], or “None” if no SDG applies.
"""


def analyze_sdgs_multiple(text, runs=3):
    """Analyze course description multiple times and return consensus"""
    if pd.isna(text) or not text.strip():
        return {
            'sdgs': '',
            'justification': '',
            'confidence': 1.0  # Added confidence
        }

    # Collect multiple analyses
    analyses = []
    for i in range(runs):
        try:
            response = client.chat.completions.create(
                model="gpt-4.1-nano-2025-04-14",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Course Description: {text}"}
                ],
                temperature=0.0,
                top_p=0.1,
                max_tokens=1000
            )
            result = response.choices[0].message.content
            parsed = parse_sdg_response(result)
            analyses.append(parsed)
            time.sleep(1)
        except Exception as e:
            print(f"Error in analysis run {i+1}: {e}")
            analyses.append({
                'sdgs': 'ERROR',
                'justification': str(e),
                'confidence': 0.0  # Added confidence
            })

    # Determine consensus
    try:
        sdg_counts = Counter([a['sdgs'] for a in analyses if 'sdgs' in a])
        if not sdg_counts:
            return {
                'sdgs': 'ERROR',
                'justification': 'All analyses failed',
                'confidence': 0.0
            }

        most_common_sdgs, count = sdg_counts.most_common(1)[0]
        confidence = count/runs

        # Select justification from the most common SDG result
        consensus_justification = next(
            (a['justification'] for a in analyses
             if a.get('sdgs') == most_common_sdgs and 'justification' in a),
            "No consensus justification"
        )

        return {
            'sdgs': most_common_sdgs,
            'justification': consensus_justification,
            'confidence': confidence
        }

    except Exception as e:
        print(f"Consensus calculation error: {e}")
        return {
            'sdgs': 'ERROR',
            'justification': f"Consensus failed: {str(e)}",
            'confidence': 0.0
        }

def parse_sdg_response(response_text):
    """Parse the structured response into components"""
    result = {
        'sdgs': '',
        'justification': ''
    }

    try:
        lines = [line.strip() for line in response_text.split('\n') if line.strip()]

        for line in lines:
            if line.startswith('SDGs:'):
                result['sdgs'] = line.replace('SDGs:', '').strip()
            elif line.startswith('Justification:'):
                result['justification'] = line.replace('Justification:', '').strip()

    except Exception as e:
        print(f"Error parsing response: {e}")

    return result

# Load the cleaned courses data
try:
    df = pd.read_csv('test2.csv', encoding='latin-1')
except FileNotFoundError:
    print("Error: cleaned_courses.csv not found. Please check the file path.")
    exit()

# Process each course description with voting system
results = []
for index, row in df.iterrows():
    print(f"Processing course {index + 1}/{len(df)}...")
    analysis = analyze_sdgs_multiple(row['Course Description'])
    results.append(analysis)
    time.sleep(1)  # Additional rate limiting between courses

# Add results to dataframe
df['sdgs'] = [r['sdgs'] for r in results]
df['justification'] = [r['justification'] for r in results]
df['confidence'] = [r['confidence'] for r in results]  # Add confidence score

# Save results
output_file = 'test1_sdgs_consensus.csv'
df.to_csv(output_file, index=False)
print(f"Analysis complete. Results saved to {output_file}")


Processing course 1/234...
Processing course 2/234...
Processing course 3/234...
Processing course 4/234...
Processing course 5/234...
Processing course 6/234...
Processing course 7/234...
Processing course 8/234...
Processing course 9/234...
Processing course 10/234...
Processing course 11/234...
Processing course 12/234...
Processing course 13/234...
Processing course 14/234...
Processing course 15/234...
Processing course 16/234...
Processing course 17/234...
Processing course 18/234...
Processing course 19/234...
Processing course 20/234...
Processing course 21/234...
Processing course 22/234...
Processing course 23/234...
Processing course 24/234...
Processing course 25/234...
Processing course 26/234...
Processing course 27/234...
Processing course 28/234...
Processing course 29/234...
Processing course 30/234...
Processing course 31/234...
Processing course 32/234...
Processing course 33/234...
Processing course 34/234...
Processing course 35/234...
Processing course 36/234...
P